# Historical experiment, not reproduced
This notebook still downloads a large dataset/model and starts training when run. Do not run it as a free hosted compute promise. Outputs were cleared. Corrected APIs were syntax-checked only; full WMT training, widgets and translation quality remain unverified. Use the local-only CLI for supplied artifacts.


In [ ]:
!pip install transformers datasets sentencepiece


In [ ]:
!pip install datasets --upgrade

In [ ]:
import torch
from transformers import MarianMTModel, MarianTokenizer, Seq2SeqTrainingArguments, Seq2SeqTrainer
from datasets import load_dataset


In [ ]:
# Specify the dataset and configuration
dataset = load_dataset("wmt14", "de-en")

# Inspect the dataset
print(dataset)


In [ ]:
# Select a smaller subset for training
small_train_dataset = dataset['train'].shuffle(seed=42).select([i for i in list(range(5000))])  # 5,000 samples
small_val_dataset = dataset['validation'].shuffle(seed=42).select([i for i in list(range(500))])  # 500 samples


In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

In [ ]:
SRC_LANG = "en"
TGT_LANG = "de"


In [ ]:
model_name = f"Helsinki-NLP/opus-mt-{SRC_LANG}-{TGT_LANG}"
tokenizer = MarianTokenizer.from_pretrained(model_name)
model = MarianMTModel.from_pretrained(model_name)
model.to(device)


In [ ]:
# Preprocessing is defined once in the next cell.


In [ ]:
def preprocess_function(examples):
    inputs = [pair[SRC_LANG] for pair in examples["translation"]]
    targets = [pair[TGT_LANG] for pair in examples["translation"]]
    return tokenizer(inputs, text_target=targets, max_length=128, truncation=True)

# Dynamic collator pads labels to -100, excluded by seq2seq loss.
from transformers import DataCollatorForSeq2Seq
collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)
tokenized_train = small_train_dataset.map(preprocess_function, batched=True, remove_columns=["translation"])
tokenized_val = small_val_dataset.map(preprocess_function, batched=True, remove_columns=["translation"])


In [ ]:
from transformers import TrainingArguments, Seq2SeqTrainingArguments, Seq2SeqTrainer

In [ ]:
training_args = Seq2SeqTrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    learning_rate=5e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    save_total_limit=2,
    predict_with_generate=True,
    logging_steps=100,
    report_to="none"
)


In [ ]:
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    processing_class=tokenizer,
    data_collator=collator,
)


In [ ]:
trainer.train()


In [ ]:
trainer.save_model("fine-tuned-marianmt-en-de")
tokenizer.save_pretrained("fine-tuned-marianmt-en-de")


In [ ]:
from transformers import MarianMTModel, MarianTokenizer

# Load the fine-tuned model and tokenizer
model_path = "fine-tuned-marianmt-en-de"
tokenizer = MarianTokenizer.from_pretrained(model_path)
model = MarianMTModel.from_pretrained(model_path)
model.to(device)


In [ ]:
def translate(text):
    # Tokenize the input text
    inputs = tokenizer([text], return_tensors="pt", max_length=128, truncation=True)
    input_ids = inputs['input_ids'].to(device)
    attention_mask = inputs['attention_mask'].to(device)

    # Generate translation using beam search
    with torch.no_grad():
        translated = model.generate(input_ids=input_ids, attention_mask=attention_mask, max_length=128, num_beams=4, early_stopping=True)

    # Decode the translated tokens
    translated_text = tokenizer.decode(translated[0], skip_special_tokens=True)
    return translated_text


In [ ]:

import ipywidgets as widgets
from IPython.display import display


# Create input text area
input_text = widgets.Textarea(
    value='',
    placeholder='Enter text to translate',
    description='English:',
    layout=widgets.Layout(width='500px', height='100px')
)

# Create output text area
output_text = widgets.Textarea(
    value='',
    placeholder='Translation will appear here',
    description='German:',
    layout=widgets.Layout(width='500px', height='100px'),
    disabled=True
)

# Create a button to trigger translation
translate_button = widgets.Button(description='Translate')

def on_translate_button_clicked(b):
    input_string = input_text.value
    if input_string:
        translated_string = translate(input_string)
        output_text.value = translated_string
    else:
        output_text.value = "Please enter text to translate."

translate_button.on_click(on_translate_button_clicked)


# Display the widgets
display(input_text)
display(translate_button)
display(output_text)